<a href="https://colab.research.google.com/github/jetnipitbank-maker/thai-bank-sentiment-analysis/blob/main/02_banks_news_scrapper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# [CELL-01]
import requests
import json
import time
import re
from bs4 import BeautifulSoup

# =============================================================================
# PART 1: Data Cleaning Module
# ส่วนการจัดเตรียมและทำความสะอาดข้อมูล (Data Pre-processing)
# =============================================================================
def clean_news_content(raw_html):
    """
    ฟังก์ชันสำหรับลบแท็ก HTML และข้อความที่ไม่เกี่ยวข้องกับเนื้อหาข่าว (Boilerplate removal)
    """
    if not raw_html:
        return ""

    # ใช้ BeautifulSoup เพื่อสกัดเฉพาะข้อความออกจาก HTML
    soup = BeautifulSoup(raw_html, "html.parser")
    text = soup.get_text(separator=" ", strip=True)

    # รายการ Regex Patterns สำหรับลบข้อความประชาสัมพันธ์และช่องทางติดตาม (CTAs)
    removal_patterns = [
        r"\w+\s+\w+\s+\d{2}\s+\w+\s+\d{4}.*",
        r"\d{1,2}:\d{2}\s+u\d{1,2}.*",
        r"ทันเกม.*",
        r"รู้ก่อนใคร.*",
        r"ติดตาม.*ทันหุ้น.*",
        r"ได้ทุกช่องทางเหล่านี้.*",
        r"ช่องทางติดตาม.*",
        r"Add Line.*",
        r"กดติดตาม.*",
        r"@thunhoon.*",
        r"www\.thunhoon\.com.*",
        r"อ่านข่าวเพิ่มเติม.*",
        r"Youtube.*",
        r"TikTok.*",
        r"Facebook.*",
        r"Twitter.*",
        r"Blockdit.*",
        r"Telegram.*",
        r"กด Like.*",
        r"Subscribe.*"
    ]

    # ดำเนินการลบข้อความตาม Pattern ที่กำหนด
    for pattern in removal_patterns:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE)

    # จัดการลบช่องว่างส่วนเกิน
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# =============================================================================
# PART 2: Core Scraper Module
# ส่วนหลักในการดึงข้อมูลและกรองธนาคารเป้าหมาย
# =============================================================================
def scrape_thunhoon_range(start_year, end_year, test_mode=False):
    """
    ฟังก์ชันดึงข้อมูลจาก API ของ Thunhoon โดยมีการกรองรายชื่อธนาคาร
    """
    target_banks = ['KBANK', 'SCB', 'BBL', 'BAY', 'TTB', 'KTB']

    # แก้ไข: เพิ่มคำว่า test ถ้ารันใน test_mode เพื่อไม่ให้ไฟล์ทับกัน
    prefix = "test_" if test_mode else ""
    filename = f"{prefix}dataset_thunhoon_Banks_{start_year}_{end_year}.jsonl"

    print(f"\n[Starting] เริ่มดึงข้อมูลช่วงปี {start_year}-{end_year} -> ไฟล์: {filename}")
    if test_mode:
        print("⚠️ สถานะ: TEST MODE (จำกัด 10 รายการ)")
    else:
        print("🚀 สถานะ: FULL EXECUTION (รันจริง)")

    api_url = "https://wp.thunhoon.com/wp-content/custom-api/my-posts.php"
    headers = {'User-Agent': 'Mozilla/5.0', 'Accept': 'application/json'}
    total_saved = 0
    page = 1

    with open(filename, "w", encoding="utf-8") as f:
        while True:
            params = {'per_page': 50, 'page': page, 'categories': 19, 'after': f'{start_year}-01-01T00:00:00', 'before': f'{end_year}-12-31T23:59:59'}
            try:
                response = requests.get(api_url, headers=headers, params=params, timeout=20)
                if response.status_code != 200: break
                data = response.json()
                if not data: break
                for article in data:
                    if test_mode and total_saved >= 10: break
                    title = clean_news_content(article.get('title', {}).get('rendered', ''))
                    body = clean_news_content(article.get('content', {}).get('rendered', ''))
                    if len(body) < 100: continue

                    tags_list = [t.get('name', '').upper() for t in article.get('tags_data', [])]
                    full_text = f"{title} {body} {' '.join(tags_list)}".upper()
                    matched_banks = [bank for bank in target_banks if bank in full_text]

                    if matched_banks:
                        for bank in set(matched_banks):
                            record = {"related_banks": bank, "date": article.get('date', ''), "title": title, "content": body, "url": f"https://thunhoon.com/article/{article.get('slug', '')}"}
                            f.write(json.dumps(record, ensure_ascii=False) + "\n")
                            total_saved += 1
                            if test_mode and total_saved >= 10: break
                print(f"  - กำลังประมวลผลหน้า {page}: สะสมแล้ว {total_saved} รายการ")
                if test_mode and total_saved >= 10: break
            except Exception: break
            page += 1
            time.sleep(0.6)
    print(f"[Success] เสร็จสิ้น {start_year}-{end_year} รวม {total_saved} รายการ")

if __name__ == "__main__":
    # ปรับเป็น True เพื่อทดสอบ, False เพื่อรันจริง
    TEST_MODE = False
    scrape_thunhoon_range(2025, 2025, test_mode=TEST_MODE)
    scrape_thunhoon_range(2023, 2024, test_mode=TEST_MODE)

In [ ]:
# [CELL-02]
import requests
import json
import time
import re
from bs4 import BeautifulSoup
import warnings
from bs4 import MarkupResemblesLocatorWarning

# ปิดการแจ้งเตือน BeautifulSoup สำหรับข้อความที่ไม่ใช่ HTML โดยตรง
warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

def clean_kaohoon_content(raw_html):
    if not raw_html: return ""
    soup = BeautifulSoup(raw_html, "html.parser")
    text = soup.get_text(separator=" ", strip=True)
    removal_patterns = [r"อ่านข่าวเพิ่มเติม.*", r"กดติดตาม.*", r"Youtube.*", r"TikTok.*", r"Facebook.*"]
    for pattern in removal_patterns:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE)
    text = re.sub(r'[\"\'“”‘’]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def scrape_kaohoon_banks_optimized(start_year, end_year, test_mode=False):
    # เพิ่มรายชื่อภาษาไทยควบคู่ไปกับตัวย่อภาษาอังกฤษ
    bank_map = [
        {"id": "KBANK", "keywords": ["KBANK", "ธนาคารกสิกรไทย", "กสิกรไทย"]},
        {"id": "SCB", "keywords": ["SCB", "ธนาคารไทยพาณิชย์", "ไทยพาณิชย์"]},
        {"id": "BBL", "keywords": ["BBL", "ธนาคารกรุงเทพ", "กรุงเทพ"]},
        {"id": "BAY", "keywords": ["BAY", "ธนาคารกรุงศรีอยุธยา", "กรุงศรี"]},
        {"id": "TTB", "keywords": ["TTB", "ธนาคารทหารไทยธนชาต", "ทหารไทยธนชาต"]},
        {"id": "KTB", "keywords": ["KTB", "ธนาคารกรุงไทย", "กรุงไทย"]}
    ]

    prefix = "test_" if test_mode else ""
    filename = f"{prefix}dataset_kaohoon_Banks_{start_year}_{end_year}.jsonl"

    print(f"\n[Starting] ดึงข้อมูล Kaohoon (Optimized Search + Thai Keywords) ปี {start_year}-{end_year}")
    api_url = "https://www.kaohoon.com/wp-json/wp/v2/posts"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

    with open(filename, "w", encoding="utf-8") as f:
        for bank_info in bank_map:
            bank_id = bank_info["id"]
            # ใช้คำค้นหาหลัก (ตัวย่อ) ในการ Search API
            search_query = bank_id
            print(f"🔍 กำลังค้นหาข้อมูลสำหรับธนาคาร: {bank_id} (รวมคำค้นภาษาไทย)...")

            page = 1
            bank_saved = 0
            seen_urls = set()

            while True:
                params = {
                    "search": search_query,
                    "per_page": 50,
                    "page": page,
                    "after": f"{start_year}-01-01T00:00:00",
                    "before": f"{end_year}-12-31T23:59:59"
                }
                try:
                    resp = requests.get(api_url, params=params, headers=headers, timeout=20)
                    if resp.status_code != 200: break

                    data = resp.json()
                    if not data or not isinstance(data, list): break

                    for item in data:
                        url = item.get('link', '')
                        if url in seen_urls: continue

                        title = clean_kaohoon_content(item.get('title', {}).get('rendered', ''))
                        body = clean_kaohoon_content(item.get('content', {}).get('rendered', ''))
                        full_text = (title + " " + body).upper()

                        # ตรวจสอบว่ามีคำสำคัญภาษาอังกฤษหรือภาษาไทยในเนื้อหาหรือไม่
                        if any(kw.upper() in full_text for kw in bank_info["keywords"]):
                            record = {
                                "related_banks": bank_id,
                                "date": item.get('date', '').split('T')[0],
                                "title": title,
                                "content": body,
                                "url": url
                            }
                            f.write(json.dumps(record, ensure_ascii=False) + "\n")
                            bank_saved += 1
                            seen_urls.add(url)
                            if test_mode and bank_saved >= 5: break

                    print(f"  - {bank_id}: พบข้อมูลในหน้า {page} (สะสม: {bank_saved})", end="\r")
                    if (test_mode and bank_saved >= 5) or len(data) < 50: break

                    page += 1
                    time.sleep(1)
                except Exception as e:
                    print(f"\n[Error] {bank_id} หน้า {page}: {e}")
                    break
            print(f"\n✅ {bank_id}: บันทึกเรียบร้อย {bank_saved} รายการ")

    print(f"\n[Finished] บันทึกไฟล์ {filename} สำเร็จ")

if __name__ == "__main__":
    TEST_MODE = False
    scrape_kaohoon_banks_optimized(2025, 2025, test_mode=TEST_MODE)
    scrape_kaohoon_banks_optimized(2023, 2024, test_mode=TEST_MODE)

In [ ]:
# [CELL-03]
import requests
import json
import time
import re
from bs4 import BeautifulSoup

def clean_hoonsmart_content(raw_html):
    if not raw_html: return ""
    soup = BeautifulSoup(raw_html, 'html.parser')
    text = soup.get_text(separator=' ', strip=True)
    # ลบลายน้ำและจัดระเบียบข้อความตามโค้ดต้นฉบับที่รันสำเร็จ
    text = re.sub(r'^HoonSmart\.com\s*>>\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'[-_—–=]{5,}', '', text)
    clean_content = re.sub(r'\s+', ' ', text).strip()
    return clean_content

def scrape_hoonsmart_banks(start_year, end_year, test_mode=False):
    bank_map = [
        {"id": "KBANK", "keywords": ["KBANK", "ธนาคารกสิกรไทย", "กสิกรไทย"]},
        {"id": "SCB", "keywords": ["SCB", "ธนาคารไทยพาณิชย์", "ไทยพาณิชย์"]},
        {"id": "BBL", "keywords": ["BBL", "ธนาคารกรุงเทพ", "กรุงเทพ"]},
        {"id": "BAY", "keywords": ["BAY", "ธนาคารกรุงศรีอยุธยา", "กรุงศรี"]},
        {"id": "TTB", "keywords": ["TTB", "ธนาคารทหารไทยธนชาต", "ทหารไทยธนชาต"]},
        {"id": "KTB", "keywords": ["KTB", "ธนาคารกรุงไทย", "กรุงไทย"]}
    ]

    prefix = "test_" if test_mode else ""
    filename = f"{prefix}dataset_hoonsmart_Banks_{start_year}_{end_year}.jsonl"

    print(f"\n🚀 [Starting] HoonSmart (Bank Filtering) ปี {start_year}-{end_year}")
    if test_mode:
        print("⚠️ สถานะ: TEST MODE (จำกัด 10 รายการ)")
    else:
        print("🚀 สถานะ: FULL EXECUTION (รันจริง)")

    api_url = "https://hoonsmart.com/wp-json/wp/v2/posts"
    total_saved = 0
    page = 1

    with open(filename, "w", encoding="utf-8") as f:
        while True:
            params = {
                "after": f"{start_year}-01-01T00:00:00",
                "before": f"{end_year}-12-31T23:59:59",
                "per_page": 100,
                "page": page,
                "_fields": "date,link,title,content"
            }

            try:
                response = requests.get(api_url, params=params, timeout=20)
                if response.status_code == 400: break
                if response.status_code != 200: break

                data = response.json()
                if not data: break

                for item in data:
                    if test_mode and total_saved >= 10: break

                    # ถอดแท็ก HTML จากหัวข้อและเนื้อหา
                    title = BeautifulSoup(item['title']['rendered'], 'html.parser').get_text(strip=True)
                    body = clean_hoonsmart_content(item['content']['rendered'])

                    # กรองข่าวสั้น (ขยะ/โพสต์คั่นหน้า)
                    if len(body) < 100: continue

                    full_text = f"{title} {body}".upper()

                    # Hybrid Filtering แยก Record รายธนาคาร
                    for bank_info in bank_map:
                        if any(kw.upper() in full_text for kw in bank_info["keywords"]):
                            record = {
                                "related_banks": bank_info["id"],
                                "date": item.get('date', '').split('T')[0],
                                "title": title,
                                "content": body,
                                "url": item.get('link', '')
                            }
                            f.write(json.dumps(record, ensure_ascii=False) + "\n")
                            total_saved += 1
                            if test_mode and total_saved >= 10: break

                print(f"  - กำลังประมวลผลหน้า {page}: สะสมแล้ว {total_saved} รายการ", end="\r")
                if test_mode and total_saved >= 10: break
                page += 1
                time.sleep(0.5)

            except Exception as e:
                print(f"\n❌ Error ที่หน้า {page}: {e}")
                break

    print(f"\n✅ เสร็จสิ้น HoonSmart {start_year}-{end_year} รวม {total_saved} รายการ")

if __name__ == "__main__":
    # ตั้งค่า TEST_MODE เป็น False เพื่อรันดึงข้อมูลฉบับเต็ม
    TEST_MODE = False
    scrape_hoonsmart_banks(2025, 2025, test_mode=TEST_MODE)
    scrape_hoonsmart_banks(2023, 2024, test_mode=TEST_MODE)

In [ ]:
# [CELL-04]
from curl_cffi import requests as cureq
from bs4 import BeautifulSoup
import json
import time
import re
import warnings
from bs4 import MarkupResemblesLocatorWarning

# ปิดการแจ้งเตือน BeautifulSoup สำหรับข้อความที่ไม่ใช่ HTML
warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

def scrape_thaipr_banks(start_year, end_year, test_mode=False):
    """
    Scraper สำหรับ ThaiPR โดยใช้โครงสร้าง API จาก [CELL-05]
    แต่เพิ่มระบบคัดกรองธนาคาร (Hybrid Filtering) ให้เหมือนกับ Scraper ตัวอื่นๆ
    """
    # รายชื่อธนาคารและ Keyword สำหรับการกรอง
    bank_map = [
        {"id": "KBANK", "keywords": ["KBANK", "ธนาคารกสิกรไทย", "กสิกรไทย"]},
        {"id": "SCB", "keywords": ["SCB", "ธนาคารไทยพาณิชย์", "ไทยพาณิชย์"]},
        {"id": "BBL", "keywords": ["BBL", "ธนาคารกรุงเทพ", "กรุงเทพ"]},
        {"id": "BAY", "keywords": ["BAY", "ธนาคารกรุงศรีอยุธยา", "กรุงศรี"]},
        {"id": "TTB", "keywords": ["TTB", "ธนาคารทหารไทยธนชาต", "ทหารไทยธนชาต"]},
        {"id": "KTB", "keywords": ["KTB", "ธนาคารกรุงไทย", "กรุงไทย"]}
    ]

    prefix = "test_" if test_mode else ""
    site_name = "ThaiPR"
    filename = f"{prefix}dataset_{site_name}_Banks_{start_year}_{end_year}.jsonl"

    print(f"\n🚀 [Starting] {site_name} (Bank Specific Filtering) ปี {start_year}-{end_year}")
    if test_mode: print("⚠️ สถานะ: TEST MODE (จำกัด 10 รายการ)")

    base_url = "https://www.thaipr.net"
    api_url = f"{base_url}/wp-json/wp/v2/posts"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

    total_saved = 0
    page = 1

    with open(filename, "w", encoding="utf-8") as f:
        while True:
            print(f"  📖 กำลังตรวจสอบหน้า {page}... (สะสมแล้ว {total_saved} ข่าว)", end="\r")

            params = {
                "per_page": 100,
                "page": page,
                "after": f"{start_year}-01-01T00:00:00",
                "before": f"{end_year}-12-31T23:59:59"
            }

            try:
                resp = cureq.get(api_url, params=params, headers=headers, impersonate="chrome", timeout=20)

                if resp.status_code in [400, 404]: break
                if resp.status_code != 200: break

                data = resp.json()
                if not data: break

                for item in data:
                    if test_mode and total_saved >= 10: break

                    # 🎯 ล็อกเป้าเฉพาะสายการเงินเพื่อลด Noise
                    link = item.get('link', '')
                    if '/finance/' not in link.lower():
                        continue

                    # สกัดเนื้อหา
                    title = BeautifulSoup(item.get('title', {}).get('rendered', ''), 'html.parser').get_text(strip=True)
                    raw_content = item.get('content', {}).get('rendered', '')
                    body = BeautifulSoup(raw_content, 'html.parser').get_text(separator=' ', strip=True)

                    # Clean Text เบื้องต้น
                    full_text = f"{title} {body}".upper()

                    # Hybrid Filtering แยก Record ตามธนาคาร
                    for bank_info in bank_map:
                        if any(kw.upper() in full_text for kw in bank_info["keywords"]):
                            record = {
                                "related_banks": bank_info["id"],
                                "date": item.get('date', '').split('T')[0],
                                "title": title,
                                "content": re.sub(r'\s+', ' ', body).strip(),
                                "url": link
                            }
                            f.write(json.dumps(record, ensure_ascii=False) + "\n")
                            total_saved += 1
                            if test_mode and total_saved >= 10: break

                if (test_mode and total_saved >= 10) or len(data) < 100:
                    break

                page += 1
                time.sleep(1)

            except Exception as e:
                print(f"\n❌ Error หน้าที่ {page}: {e}")
                break

    print(f"\n✅ เสร็จสิ้น ThaiPR {start_year}-{end_year} รวม {total_saved} รายการ บันทึกที่: {filename}")

if __name__ == "__main__":
    # ตั้งค่า TEST_MODE เป็น False เพื่อรันดึงข้อมูลฉบับเต็ม
    TEST_MODE = False
    scrape_thaipr_banks(2025, 2025, test_mode=TEST_MODE)
    scrape_thaipr_banks(2023, 2024, test_mode=TEST_MODE)

In [ ]:
# [CELL-05]
from curl_cffi import requests as cureq
from bs4 import BeautifulSoup
import json
import time
import re

def scrape_money_banking_optimized(start_year, end_year, test_mode=False):
    """
    Optimized scraper for Money & Banking using validated Category IDs (102, 105)
    """
    site_name = "Money_Banking"
    base_url = "https://moneyandbanking.co.th"
    target_category_ids = "102,105"

    bank_map = [
        {"id": "KBANK", "keywords": ["KBANK", "ธนาคารกสิกรไทย", "กสิกรไทย"]},
        {"id": "SCB", "keywords": ["SCB", "ธนาคารไทยพาณิชย์", "ไทยพาณิชย์"]},
        {"id": "BBL", "keywords": ["BBL", "ธนาคารกรุงเทพ", "กรุงเทพ"]},
        {"id": "BAY", "keywords": ["BAY", "ธนาคารกรุงศรีอยุธยา", "กรุงศรี"]},
        {"id": "TTB", "keywords": ["TTB", "ธนาคารทหารไทยธนชาต", "ทหารไทยธนชาต"]},
        {"id": "KTB", "keywords": ["KTB", "ธนาคารกรุงไทย", "กรุงไทย"]}
    ]

    prefix = "test_" if test_mode else ""
    filename = f"{prefix}dataset_{site_name}_Banks_{start_year}_{end_year}.jsonl"

    print(f"\n🚀 [WP API] Target: {site_name} (Categories: {target_category_ids}) | Year: {start_year}-{end_year}")

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "application/json, text/plain, */*",
        "Referer": base_url
    }

    api_url = f"{base_url}/wp-json/wp/v2/posts"
    page = 1
    total_saved = 0

    with open(filename, "w", encoding="utf-8") as f_out:
        while True:
            print(f"  📖 Page {page} | Total Saved: {total_saved}", end="\r")
            params = {
                "per_page": 50,
                "page": page,
                "after": f"{start_year}-01-01T00:00:00",
                "before": f"{end_year}-12-31T23:59:59",
                "categories": target_category_ids
            }

            try:
                resp = cureq.get(api_url, params=params, headers=headers, impersonate="chrome110", timeout=30)

                if resp.status_code in [400, 404]: break
                if resp.status_code != 200: break

                data = resp.json()
                if not data or not isinstance(data, list): break

                for item in data:
                    title = BeautifulSoup(item.get('title', {}).get('rendered', ''), 'html.parser').get_text(strip=True)
                    body_raw = item.get('content', {}).get('rendered', '')
                    body = BeautifulSoup(body_raw, 'html.parser').get_text(separator=' ', strip=True)

                    full_text = f"{title} {body}".upper()

                    # Hybrid Filtering for specific banks
                    matched_banks = []
                    for bank_info in bank_map:
                        if any(kw.upper() in full_text for kw in bank_info["keywords"]):
                            matched_banks.append(bank_info["id"])

                    if matched_banks:
                        for b_id in set(matched_banks):
                            record = {
                                "related_banks": b_id,
                                "date": item.get('date', '').split('T')[0],
                                "url": item.get('link', ''),
                                "title": title,
                                "content": re.sub(r'\s+', ' ', body).strip()
                            }
                            f_out.write(json.dumps(record, ensure_ascii=False) + "\n")
                            total_saved += 1

                if len(data) < 50: break
                page += 1
                time.sleep(1.2)
            except Exception as e:
                print(f"\n❌ Error: {e}")
                break

    print(f"\n✅ Finished {site_name}! Total saved: {total_saved} articles stored in {filename}")

if __name__ == "__main__":
    # รันดึงข้อมูลจริง
    scrape_money_banking_optimized(2025, 2025)
    scrape_money_banking_optimized(2023, 2024)

In [ ]:
# [CELL-05.1] HoonVision Recovery Scraper (Enhanced Logging)
import json
import time
import re
import sys
from curl_cffi import requests as cureq
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

def scrape_hoonvision_recovery(start_year, end_year):
    """
    HoonVision Scraper: ดึงข้อมูลผ่าน WP-JSON API พร้อมระบบ Logging และความปลอดภัยสูง
    """
    bank_map = [
        {"id": "KBANK", "keywords": ["KBANK", "ธนาคารกสิกรไทย", "กสิกรไทย"]},
        {"id": "SCB", "keywords": ["SCB", "ธนาคารไทยพาณิชย์", "ไทยพาณิชย์"]},
        {"id": "BBL", "keywords": ["BBL", "ธนาคารกรุงเทพ", "กรุงเทพ"]},
        {"id": "BAY", "keywords": ["BAY", "ธนาคารกรุงศรีอยุธยา", "กรุงศรี"]},
        {"id": "TTB", "keywords": ["TTB", "ธนาคารทหารไทยธนชาต", "ทหารไทยธนชาต"]},
        {"id": "KTB", "keywords": ["KTB", "ธนาคารกรุงไทย", "กรุงไทย"]}
    ]

    filename = f"dataset_hoonvision_Banks_{start_year}_{end_year}.jsonl"
    api_url = "https://hoonvision.com/wp-json/wp/v2/posts"
    session = cureq.Session(impersonate="chrome110", timeout=30)

    print(f"\n🚀 [HoonVision] Starting API Recovery: {start_year}-{end_year}")
    print(f"📂 Output File: {filename}")

    total_saved = 0
    page = 1

    with open(filename, "w", encoding="utf-8") as f:
        # ใช้ while True เพื่อดึงข้อมูลจนกว่าจะหมดหน้า (Pagination)
        while True:
            params = {
                "per_page": 50,
                "page": page,
                "after": f"{start_year}-01-01T00:00:00",
                "before": f"{end_year}-12-31T23:59:59",
                "_fields": "date,link,title,content"
            }

            try:
                resp = session.get(api_url, params=params)

                if resp.status_code == 400: # กรณีระบุหน้าเกินจริง
                    print(f"\n🏁 Reached the end of available data at page {page-1}")
                    break
                if resp.status_code != 200:
                    print(f"\n⚠️ Unexpected status code {resp.status_code} at page {page}")
                    break

                data = resp.json()
                if not data: break

                page_saved = 0
                for item in data:
                    # สกัด Title และ Body (แก้ปัญหา AJAX content หาย)
                    title = BeautifulSoup(item['title']['rendered'], 'html.parser').get_text(strip=True)
                    body_raw = item['content']['rendered']
                    body = BeautifulSoup(body_raw, 'html.parser').get_text(separator=' ', strip=True)

                    # ขจัด Noise และความยาวขั้นต่ำ
                    if len(body) < 150: continue

                    full_text = f"{title} {body}".upper()
                    matched = [b['id'] for b in bank_map if any(kw.upper() in full_text for kw in b['keywords'])]

                    if matched:
                        for b_id in set(matched):
                            record = {
                                "related_banks": b_id,
                                "date": item['date'][:10],
                                "title": title,
                                "content": re.sub(r'\s+', ' ', body).strip(),
                                "url": item['link']
                            }
                            f.write(json.dumps(record, ensure_ascii=False) + "\n")
                            total_saved += 1
                            page_saved += 1

                # Logging สถานะการทำงาน
                sys.stdout.write(f"\r   📖 Processing Page {page} | Saved in this page: {page_saved} | Cumulative Total: {total_saved} articles")
                sys.stdout.flush()

                page += 1
                time.sleep(1.2) # Avoid rate limiting

            except Exception as e:
                print(f"\n❌ Network Error at page {page}: {e}")
                break

    print(f"\n✅ [Success] HoonVision {start_year}-{end_year} Finished. Total saved: {total_saved} records.\n")

if __name__ == "__main__":
    # ดึงข้อมูลปีปัจจุบัน 2025
    scrape_hoonvision_recovery(2025, 2025)
    # ดึงข้อมูลย้อนหลัง 2023-2024
    scrape_hoonvision_recovery(2023, 2024)

In [ ]:
# [CELL-06]
from curl_cffi import requests as cureq
from bs4 import BeautifulSoup
import json
import time
import re
import warnings
from bs4 import MarkupResemblesLocatorWarning

warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

def scrape_infoquest_banks_production(start_year=2023, end_year=2025, test_mode=False):
    print(f"\n🚀 [INFOQUEST BANK-FILTER] เริ่มการดึงข้อมูลและคัดกรองธนาคาร ปี {start_year}-{end_year}")
    print(f"{'='*70}")

    # รายชื่อธนาคารและ Keyword สำหรับการคัดกรอง (Hybrid Filtering)
    bank_map = [
        {"id": "KBANK", "keywords": ["KBANK", "กสิกรไทย"]},
        {"id": "SCB", "keywords": ["SCB", "ไทยพาณิชย์"]},
        {"id": "BBL", "keywords": ["BBL", "ธนาคารกรุงเทพ"]},
        {"id": "BAY", "keywords": ["BAY", "กรุงศรี"]},
        {"id": "TTB", "keywords": ["TTB", "ทหารไทยธนชาต"]},
        {"id": "KTB", "keywords": ["KTB", "กรุงไทย"]}
    ]

    prefix = "test_" if test_mode else ""
    filename = f"{prefix}dataset_infoquest_Banks_{start_year}_{end_year}.jsonl"
    with open(filename, "w", encoding="utf-8") as f: pass

    # 🗺️ 1. Scan Sitemaps (ล็อกเป้าแฟ้มที่ครอบคลุมช่วงปี 2023-2025)
    locked_sitemaps = [f"https://www.infoquest.co.th/post-sitemap{i}.xml" for i in range(60, 160)]
    target_links = set()

    print(f"🔍 1. สแกน Sitemap เพื่อค้นหาลิงก์ข่าวปี {start_year}-{end_year}...")
    for sm_url in locked_sitemaps:
        try:
            sm_resp = cureq.get(sm_url, impersonate="chrome", timeout=15)
            if sm_resp.status_code != 200: continue

            sm_soup = BeautifulSoup(sm_resp.text, 'xml')
            urls = sm_soup.find_all('loc')

            for url_tag in urls:
                link = url_tag.text
                year_match = re.search(r'/(\d{4})/', link)
                if year_match and start_year <= int(year_match.group(1)) <= end_year:
                    target_links.add(link)
        except Exception: continue

    print(f"🎯 พบเป้าหมายเบื้องต้น {len(target_links)} ลิงก์")

    # 🚜 2. Content Extraction & Bank Filtering
    print(f"🚜 2. กำลังดึงเนื้อหาและกรองธนาคาร... (บันทึกที่: {filename})")
    total_saved = 0
    processed = 0

    for link in target_links:
        processed += 1
        if test_mode and total_saved >= 10: break

        try:
            art_resp = cureq.get(link, impersonate="chrome", timeout=15)
            if art_resp.status_code == 200:
                art_soup = BeautifulSoup(art_resp.text, 'html.parser')

                # สกัดเนื้อหาและหัวข้อ
                title_tag = art_soup.find('h1') or art_soup.find('title')
                title = title_tag.get_text(strip=True) if title_tag else ""

                paragraphs = art_soup.find_all('p')
                raw_content = ' '.join([p.get_text(strip=True) for p in paragraphs])

                # 🧹 Data Cleansing (Logic from CELL-09)
                clean_content = re.sub(r'https?://\S+|www\.\S+', '', raw_content)
                clean_content = re.sub(r'(?:สำนักข่าวอินโฟเควสท์|โดย สำนักข่าวอินโฟเควสท์|อินโฟเควสท์|InfoQuest|Dataxet|ดาต้าเซ็ต)', '', clean_content, flags=re.IGNORECASE)

                # ตัดขยะโฆษณา Dataxet และวงเล็บวันที่ท้ายข่าว
                for noise in ["อัปเดตข่าวสารด้านเศรษฐกิจ", "บริษัท ดาต้าเซ็ต จำกัด"]:
                    if noise in clean_content: clean_content = clean_content.split(noise)[0]

                clean_content = re.sub(r'\(\d+\s*[ก-ฮ]+\.[ก-ฮ]+\.\s*\d+\)\s*$', '', clean_content).strip()
                clean_content = re.sub(r'\s+', ' ', clean_content).strip()

                if len(clean_content) < 50: continue

                # 🔍 Bank Filtering (Hybrid Mode)
                full_text = f"{title} {clean_content}".upper()
                matched_banks = []
                for bank_info in bank_map:
                    if any(kw.upper() in full_text for kw in bank_info["keywords"]):
                        matched_banks.append(bank_info["id"])

                if matched_banks:
                    date_tag = art_soup.find('meta', property='article:published_time')
                    pub_date = date_tag['content'][:10] if date_tag else ""

                    for b_id in set(matched_banks):
                        record = {
                            "related_banks": b_id,
                            "date": pub_date,
                            "url": link,
                            "title": title,
                            "content": clean_content
                        }
                        with open(filename, "a", encoding="utf-8") as f_out:
                            f_out.write(json.dumps(record, ensure_ascii=False) + "\n")
                        total_saved += 1

            if processed % 10 == 0:
                print(f"  - ประมวลผล {processed}/{len(target_links)} | บันทึกแล้ว {total_saved} รายการ", end="\r")

        except Exception: continue

    print(f"\n\n✅ ภารกิจสำเร็จ! บันทึกข้อมูล InfoQuest: {total_saved} รายการ")

if __name__ == "__main__":
    # แนะนำให้รันปี 2025 ก่อน แล้วตามด้วย 2023-2024
    scrape_infoquest_banks_production(start_year=2025, end_year=2025, test_mode=False)
    scrape_infoquest_banks_production(start_year=2023, end_year=2024, test_mode=False)

In [ ]:
# [CELL-07]
import json
import os
import re
import glob

def refine_infoquest_data(input_pattern="dataset_infoquest_Banks_*.jsonl"):
    """
    ฟังก์ชันสำหรับคลีนข้อมูล InfoQuest ในระดับ Production
    เพิ่มการกรองลิงก์รูปภาพและไฟล์แนบ (dxt-content) ที่ไม่ใช่เนื้อหาข่าวออก
    """
    files = glob.glob(input_pattern)
    if not files:
        print("❌ ไม่พบไฟล์ InfoQuest ที่ต้องการคลีน")
        return

    # Patterns สำหรับการคลีนเนื้อหา
    cleaning_patterns = [
        (r'\(\d+\s*[ก-ฮ]+\.[ก-ฮ]+\.\s*\d+\)\s*$', ''), # ลบ (26 ม.ค. 66) ท้ายข่าว
        (r'--+\s*อินโฟเควสท์', ''),
        (r'\s*โดย\s*\w+\s*\w+.*$', ''),
        (r'อ่านรายละเอียดเพิ่มเติม.*', ''),
        (r'คลิกที่นี่เพื่อ.*', ''),
        (r'\s*\(จบ\)\s*', '')
    ]

    for file_path in files:
        refined_records = []
        seen_signatures = set()
        trash_count = 0

        print(f"🧹 กำลังคลีนไฟล์: {file_path}")

        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    item = json.loads(line)
                    url = item.get('url', '').lower()
                    content = item.get('content', '')
                    title = item.get('title', '')

                    # 1. 🛑 Attachment Filter: ลบลิงก์รูปภาพหรือไฟล์แนบ Dataxet ทิ้ง
                    # หากเป็น URL ที่ชี้ไปยังโฟลเดอร์ uploads หรือมีนามสกุลภาพ จะไม่นำมาประมวลผล
                    if "dxt-content/uploads" in url or url.endswith(('.jpg', '.jpeg', '.png', '.gif')):
                        trash_count += 1
                        continue

                    # 2. ล้าง Noise ในเนื้อหา
                    for pattern, replacement in cleaning_patterns:
                        content = re.sub(pattern, replacement, content, flags=re.IGNORECASE)

                    content = re.sub(r'\s+', ' ', content).strip()

                    # 3. Quality Filter (สกัดข่าวที่ไม่มีเนื้อหาจริงออก)
                    if len(content) < 100 or not title:
                        continue

                    # 4. Deduplication
                    sig = f"{item['url']}_{item['related_banks']}"
                    if sig in seen_signatures:
                        continue
                    seen_signatures.add(sig)

                    item['content'] = content
                    refined_records.append(item)
                except Exception:
                    continue

        # บันทึกไฟล์ที่คลีนแล้ว
        output_path = file_path.replace(".jsonl", "_Refined.jsonl")
        with open(output_path, 'w', encoding='utf-8') as f_out:
            for record in refined_records:
                f_out.write(json.dumps(record, ensure_ascii=False) + "\n")

        print(f"✅ คลีนเสร็จแล้ว: {len(refined_records)} รายการ (ลบทิ้ง {trash_count} ลิงก์ขยะ) -> {output_path}")

if __name__ == "__main__":
    refine_infoquest_data()

In [ ]:
# [CELL-08] PPTV RECOVERY: Stable Sequential Scraper (V12-Based with Detailed Logs)
import json
import time
import random
import re
from datetime import date, timedelta
from curl_cffi import requests as cureq
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

def scrape_pptv_v12_stable(start_year, end_year):
    print(f"\n{'='*30} STARTING RECOVERY: {start_year}-{end_year} {'='*30}")

    bank_map = [
        {"id": "KBANK", "keywords": ["KBANK", "ธนาคารกสิกรไทย", "กสิกรไทย"]},
        {"id": "SCB", "keywords": ["SCB", "ธนาคารไทยพาณิชย์", "ไทยพาณิชย์"]},
        {"id": "BBL", "keywords": ["BBL", "ธนาคารกรุงเทพ", "กรุงเทพ"]},
        {"id": "BAY", "keywords": ["BAY", "ธนาคารกรุงศรีอยุธยา", "กรุงศรี"]},
        {"id": "TTB", "keywords": ["TTB", "ธนาคารทหารไทยธนชาต", "ทหารไทยธนชาต"]},
        {"id": "KTB", "keywords": ["KTB", "ธนาคารกรุงไทย", "กรุงไทย"]}
    ]

    target_categories = ['/wealth/stock-investment/', '/wealth/monetary/', '/news/economy/']
    filename = f"dataset_pptv_stable_v12_{start_year}_{end_year}.jsonl"

    s_date = date(start_year, 1, 1)
    e_date = date(end_year, 12, 31)
    days = [s_date + timedelta(n) for n in range((e_date - s_date).days + 1)]

    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"}
    total_saved = 0

    with open(filename, "w", encoding="utf-8") as f_out:
        for current_date in tqdm(days, desc=f"Processing {start_year}"):
            date_str = current_date.strftime("%Y-%m-%d")
            sitemap_url = f"https://www.pptvhd36.com/sitemap-{date_str}.xml"

            try:
                # Step 1: Fetch Sitemap
                resp = cureq.get(sitemap_url, impersonate="chrome110", timeout=20)
                if resp.status_code != 200:
                    continue

                soup = BeautifulSoup(resp.text, 'xml')
                links = [loc.text for loc in soup.find_all('loc') if any(cat in loc.text for cat in target_categories)]

                if links:
                    print(f"\n[Sitemap] {date_str}: Found {len(links)} potential links")

                for link in links:
                    # Step 2: Human-like delay
                    time.sleep(random.uniform(1.5, 2.8))

                    art_resp = cureq.get(link, impersonate="chrome110", headers=headers, timeout=25)
                    if art_resp.status_code == 200:
                        art_soup = BeautifulSoup(art_resp.text, 'html.parser')
                        title = art_soup.find('h1').get_text(strip=True) if art_soup.find('h1') else ""
                        body = ' '.join([p.get_text(strip=True) for p in art_soup.find_all('p')])

                        # Step 3: Bank Filtering
                        full_text = f"{title} {body}".upper()
                        matched = [b['id'] for b in bank_map if any(kw.upper() in full_text for kw in b['keywords'])]

                        if matched:
                            for b_id in set(matched):
                                record = {
                                    "related_banks": b_id,
                                    "date": date_str,
                                    "url": link,
                                    "title": title,
                                    "content": re.sub(r'\s+', ' ', body).strip()
                                }
                                f_out.write(json.dumps(record, ensure_ascii=False) + "\n")
                                total_saved += 1
                            print(f"  ✨ Saved: [{matched}] {title[:50]}... (Total: {total_saved})", end='\r')

            except Exception as e:
                print(f"\n⚠️ Error on {date_str}: {str(e)}")
                time.sleep(5)
                continue

    print(f"\n\n✅ COMPLETED: {filename} | Total Saved: {total_saved} articles")

if __name__ == "__main__":
    # รันแยกเป็นปีเพื่อความชัดเจนของไฟล์
    scrape_pptv_v12_stable(2025, 2025)
    scrape_pptv_v12_stable(2023, 2024)

In [ ]:
# [CELL-09] PPTV Data Refinement & Deduplication (Ultra Clean V3: Advanced Noise Removal)
import json
import os
import re
import glob

def refine_pptv_recovery_data(input_pattern="dataset_pptv_stable_v12_*.jsonl"):
    """
    ฟังก์ชันสำหรับคลีนข้อมูล PPTV ขั้นสูง กำจัดคำซ้ำช่วงต้นข่าว Footer และข้อความแจ้งเตือนคุกกี้แบบยาว
    """
    files = glob.glob(input_pattern)
    if not files:
        print("❌ ไม่พบไฟล์ PPTV Stable V12 สำหรับการคลีน")
        return

    # ข้อความคุกกี้ที่ผู้ใช้ระบุให้ตัดออก
    cookie_consent_text = r"เว็บไซต์ของเราใช้คุกกี้เพื่อให้ทำงานได้อย่างสมบูรณ์.*?ไม่สามารถระบุตัวตนได้"

    # Patterns สำหรับการคลีนเนื้อหา
    cleaning_patterns = [
        (r'^.*?\(PPTV\)\s?-?\s?', ''),
        (r'^เมื่อวันที่\s?\d+.*?(ผู้สื่อข่าวรายงานว่า|รายงานว่า)', ''),
        (r'^ผู้สื่อข่าวรายงานว่า', ''),
        (cookie_consent_text, ''), # เพิ่มการลบข้อความคุกกี้แบบละเอียด
        (r'โดย\s?PPTV\s?Online', ''),
        (r'คุณสามารถปรับแต่งคุกกี้ได้ที่นี่', ''),
        (r'คอนเทนต์แนะนำ', ''),
        (r'อัตราแลกเปลี่ยนถัวเฉลี่ยถ่วงน้ำหนัก', ''),
        (r'อ่านข่าวเพิ่มเติม.*', ''),
        (r'สรุปข่าว.*', ''),
        (r'วิดีโอที่เกี่ยวข้อง', ''),
        (r'ภาพ\s?:\s?.*', ''),
        (r'Credit\s?Video\s?:\s?.*', ''),
        (r'ขอบคุณข้อมูลจาก\s?:\s?.*', ''),
        (r'ติดตามข่าว.*PPTV.*', ''),
        (r'#\w+', ''),
        (r'[\"\'“”‘’]', '')
    ]

    for file_path in files:
        refined_records = []
        seen_signatures = set()

        year_match = re.search(r'(\d{4}_\d{4})', file_path)
        year_range = year_match.group(1) if year_match else "Unknown"

        print(f"🧹 Ultra Cleaning (V3) PPTV File: {file_path}")

        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    item = json.loads(line)
                    content = item.get('content', '')
                    title = item.get('title', '')

                    # 1. ล้าง Noise ตาม Pattern
                    for pattern, replacement in cleaning_patterns:
                        content = re.sub(pattern, replacement, content, count=1 if '^' in pattern else 0, flags=re.IGNORECASE | re.DOTALL)

                    # 2. จัดการ White Spaces และ Hidden Characters
                    content = content.replace('\u200b', '')
                    content = re.sub(r'^[\s\-\.\|\:]+', '', content)
                    content = re.sub(r'\s+', ' ', content).strip()
                    title = re.sub(r'[\"\'“”\'‘’]', '', title).strip()

                    # 3. Quality Filter
                    if len(content) < 150 or not title:
                        continue

                    # 4. Deduplication
                    sig = f"{item['url']}_{item['related_banks']}"
                    if sig in seen_signatures:
                        continue
                    seen_signatures.add(sig)

                    item['content'] = content
                    item['title'] = title
                    refined_records.append(item)
                except Exception:
                    continue

        output_path = f"dataset_pptv_Banks_{year_range}_Refined.jsonl"
        with open(output_path, 'w', encoding='utf-8') as f_out:
            for record in refined_records:
                f_out.write(json.dumps(record, ensure_ascii=False) + "\n")

        print(f"✅ คลีนเสร็จสมบูรณ์ (V3): {len(refined_records)} รายการ -> {output_path}")

if __name__ == "__main__":
    refine_pptv_recovery_data()

In [ ]:
# [CELL-10]
import asyncio
import sys
import json
import re
import time
import os
import subprocess

# =============================================================================
# Business Today Production-Grade Scraper (V48: Multi-Category & Real-time Logs)
# =============================================================================

script_content = r"""
import asyncio
import sys
import json
import re
from playwright.async_api import async_playwright

async def scrape_year_range(start_year, end_year, page_range):
    bank_map = [
        {'id': 'KBANK', 'keywords': ['KBANK', 'ธนาคารกสิกรไทย', 'กสิกรไทย']},
        {'id': 'SCB', 'keywords': ['SCB', 'ธนาคารไทยพาณิชย์', 'ไทยพาณิชย์']},
        {'id': 'BBL', 'keywords': ['BBL', 'ธนาคารกรุงเทพ', 'กรุงเทพ']},
        {'id': 'BAY', 'keywords': ['BAY', 'ธนาคารกรุงศรีอยุธยา', 'กรุงศรี']},
        {'id': 'TTB', 'keywords': ['TTB', 'ธนาคารทหารไทยธนชาต', 'ทหารไทยธนชาต']},
        {'id': 'KTB', 'keywords': ['KTB', 'ธนาคารกรุงไทย', 'กรุงไทย']}
    ]

    filename = f'dataset_businesstoday_Banks_{start_year}_{end_year}.jsonl'
    # เพิ่ม Category Business ตามความต้องการของผู้ใช้
    categories = [
        {"name": "Stock", "url": "https://www.businesstoday.co/category/stock/"},
        {"name": "Business", "url": "https://www.businesstoday.co/category/business/"}
    ]

    print(f"\n{'='*75}")
    print(f"🚀 [V48] Business Today Scraper | Target: {start_year}-{end_year}")
    print(f"📂 Output: {filename}")
    print(f"📦 Categories: {[c['name'] for c in categories]}")
    print(f"{'='*75}")
    sys.stdout.flush()

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True, args=["--disable-blink-features=AutomationControlled"])
        context = await browser.new_context(
            user_agent='Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36',
            viewport={'width': 1280, 'height': 1200}
        )
        page = await context.new_page()
        total_saved = 0

        for cat in categories:
            print(f"\n📂 Switching to Category: {cat['name']}")
            sys.stdout.flush()

            for p_idx in page_range:
                url = f"{cat['url']}page/{p_idx}/"
                print(f"📖 [{cat['name']} - Page {p_idx}] Scanning: {url}")
                sys.stdout.flush()

                try:
                    await page.goto(url, wait_until='load', timeout=60000)
                    await asyncio.sleep(2)

                    links = await page.evaluate('''() => {
                        return Array.from(document.querySelectorAll(".td-module-title a, article a"))
                                    .map(a => a.href)
                                    .filter(h => h && /\\/\\d{2}\\/\\d{2}\\/\\d{4}\\//.test(h));
                    }''')

                    unique_links = list(set(links))
                    if not unique_links: continue

                    for art_url in unique_links:
                        date_match = re.search(r'/(\d{2})/(\d{2})/(\d{4})/', art_url)
                        if not date_match: continue
                        day, month, year = date_match.groups()
                        if not (start_year <= int(year) <= end_year): continue

                        try:
                            art_page = await context.new_page()
                            await art_page.goto(art_url, wait_until='domcontentloaded', timeout=30000)
                            data = await art_page.evaluate('''() => {
                                const title = document.querySelector("h1, .entry-title, .tdb-title-text")?.innerText || "";
                                const paragraphs = Array.from(document.querySelectorAll(".td-post-content p, .entry-content p, p"))
                                                .map(p => p.innerText.trim())
                                                .filter(t => t.length > 30);
                                let content = paragraphs.join(" ");
                                return { title: title.trim(), content: content.trim() };
                            }''')

                            if len(data['content']) < 150:
                                await art_page.close()
                                continue

                            full_text = f"{data['title']} {data['content']}".upper()
                            matched_banks = [b['id'] for b in bank_map if any(kw.upper() in full_text for kw in b['keywords'])]

                            if matched_banks:
                                for b_id in set(matched_banks):
                                    record = {
                                        "related_banks": b_id, "date": f"{year}-{month}-{day}",
                                        "url": art_url, "title": data['title'],
                                        "content": data['content'].replace('\u200b', '').strip(),
                                        "category": cat['name']
                                    }
                                    with open(filename, "a", encoding="utf-8") as f:
                                        f.write(json.dumps(record, ensure_ascii=False) + "\n")
                                    total_saved += 1
                                print(f"      ✨ Saved: {data['title'][:40]}... (Total: {total_saved})")
                                sys.stdout.flush()
                            await art_page.close()
                        except:
                            if 'art_page' in locals(): await art_page.close()
                except Exception as e:
                    print(f"   ❌ Error on Page {p_idx}: {e}")
                    sys.stdout.flush()

        await browser.close()
        print(f"\n🏁 Range {start_year}-{end_year} Complete. Total Saved: {total_saved}")
        sys.stdout.flush()

async def main():
    # ลำดับการรัน: ปีปัจจุบัน (หน้า 1-50) แล้วตามด้วย Archive (หน้า 51-180)
    await scrape_year_range(2025, 2025, range(1, 51))
    await scrape_year_range(2023, 2024, range(51, 181))

if __name__ == '__main__':
    asyncio.run(main())
"""

with open("bt_stream_worker_v48.py", "w", encoding="utf-8") as f:
    f.write(script_content)

print("⌛ Launching Business Today Scraper V48 (Multi-Category + Streaming Logs Enabled)...")
env = os.environ.copy()
env["PYTHONIOENCODING"] = "utf-8"

# Use Popen to stream logs in real-time to Colab
process = subprocess.Popen(
    [sys.executable, "bt_stream_worker_v48.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    env=env,
    bufsize=1
)

try:
    for line in iter(process.stdout.readline, ""):
        print(line, end="")
        sys.stdout.flush()
except KeyboardInterrupt:
    print("\n🛑 Scraper Interrupted by User. Terminating...")
    process.terminate()
finally:
    process.wait()
    print("\n✅ Scraper process finished.")

In [ ]:
# [CELL-11]
# Amarin TV Scraper with Precise Date Extraction & Bank Filtering
import requests
from curl_cffi import requests as cureq
from bs4 import BeautifulSoup
import json
import time
import re
import warnings
import sys
from bs4 import MarkupResemblesLocatorWarning

# ปิดการแจ้งเตือน BeautifulSoup สำหรับข้อความที่ไม่ใช่ HTML
warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

def scrape_amarin_banks(start_year, end_year, test_mode=False):
    """
    Scraper สำหรับ Amarin TV (Spotlight Finance)
    แก้ไขการดึงวันที่ให้เป็น Full Date (YYYY-MM-DD) แทนการใช้ค่าคงที่วันที่ 1
    """
    bank_map = [
        {"id": "KBANK", "keywords": ["KBANK", "ธนาคารกสิกรไทย", "กสิกรไทย"]},
        {"id": "SCB", "keywords": ["SCB", "ธนาคารไทยพาณิชย์", "ไทยพาณิชย์"]},
        {"id": "BBL", "keywords": ["BBL", "ธนาคารกรุงเทพ", "กรุงเทพ"]},
        {"id": "BAY", "keywords": ["BAY", "ธนาคารกรุงศรีอยุธยา", "กรุงศรี"]},
        {"id": "TTB", "keywords": ["TTB", "ธนาคารทหารไทยธนชาต", "ทหารไทยธนชาต"]},
        {"id": "KTB", "keywords": ["KTB", "ธนาคารกรุงไทย", "กรุงไทย"]}
    ]

    prefix = "test_" if test_mode else ""
    filename = f"{prefix}dataset_amarin_Banks_{start_year}_{end_year}.jsonl"

    print(f"\n{'='*80}")
    print(f"🚀 [START] Amarin TV Spotlight Finance Scraper | Years: {start_year}-{end_year}")
    print(f"📂 Output File: {filename}")
    print(f"{'='*80}\n")

    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
    url_keywords = ['finance']
    total_saved = 0
    total_processed = 0

    with open(filename, "w", encoding="utf-8") as f_out:
        for year in range(start_year, end_year + 1):
            for month in range(1, 13):
                sitemap_url = f"https://www.amarintv.com/sitemap/spotlight-{year}-{month:02d}.xml"

                try:
                    sm_resp = cureq.get(sitemap_url, headers=headers, impersonate="chrome", timeout=15)
                    if sm_resp.status_code != 200: continue

                    soup = BeautifulSoup(sm_resp.text, 'xml')
                    urls = [loc.text for loc in soup.find_all('loc') if any(kw in loc.text.lower() for kw in url_keywords)]

                    if not urls: continue

                    print(f"🔍 [Sitemap] {year}-{month:02d} | Found {len(urls)} links.")

                    for idx, link in enumerate(urls, 1):
                        if test_mode and total_saved >= 10: break
                        total_processed += 1

                        sys.stdout.write(f"\r      [Processing] {idx}/{len(urls)} | Saved: {total_saved}")
                        sys.stdout.flush()

                        try:
                            art_resp = cureq.get(link, headers=headers, impersonate="chrome", timeout=15)
                            if art_resp.status_code != 200: continue

                            art_soup = BeautifulSoup(art_resp.text, 'html.parser')

                            # --- Date Extraction Logic (Improved) ---
                            # 1. พยายามดึงจาก meta tag โดยตรง
                            pub_date_tag = art_soup.find('meta', property='article:published_time') or \
                                           art_soup.find('meta', itemprop='datePublished')

                            full_article_date = None
                            if pub_date_tag and pub_date_tag.get('content'):
                                full_article_date = pub_date_tag['content'].split('T')[0]

                            # 2. ถ้าไม่เจอ ให้ดึงจาก <time class="time-created"> (Amarin Specific)
                            if not full_article_date:
                                time_tag = art_soup.find('time', class_='time-created')
                                if time_tag and time_tag.get('datetime'):
                                    full_article_date = time_tag['datetime'].split(' ')[0]

                            # 3. Fallback (กรณีสุดวิสัยจริงๆ)
                            if not full_article_date:
                                full_article_date = f"{year}-{month:02d}-01"

                            # --- Content Extraction ---
                            title_tag = art_soup.find('h1') or art_soup.find('title')
                            clean_title = title_tag.get_text(strip=True) if title_tag else ""

                            paragraphs = [p.get_text(strip=True) for p in art_soup.find_all('p') if len(p.get_text(strip=True)) > 20]
                            clean_content = re.sub(r'\s+', ' ', ' '.join(paragraphs)).strip()

                            if len(clean_content) < 100: continue

                            full_text = f"{clean_title} {clean_content}".upper()

                            for bank_info in bank_map:
                                if any(kw.upper() in full_text for kw in bank_info["keywords"]):
                                    record = {
                                        "related_banks": bank_info["id"],
                                        "date": full_article_date,
                                        "title": clean_title,
                                        "content": clean_content,
                                        "url": link
                                    }
                                    f_out.write(json.dumps(record, ensure_ascii=False) + "\n")
                                    total_saved += 1
                                    if test_mode and total_saved >= 10: break

                        except Exception: continue
                    print(f"\n   ✅ Month {year}-{month:02d} completed.")

                except Exception: continue

    print(f"\n🏁 [FINISH] Total Saved: {total_saved} records to {filename}")

if __name__ == "__main__":
    scrape_amarin_banks(2025, 2025, test_mode=False)
    scrape_amarin_banks(2023, 2024, test_mode=False)

In [ ]:
# [CELL-12]
# Thairath Scraper with Precise Date Extraction (YYYY-MM-DD) & Bank Filtering
from curl_cffi import requests as cureq
from bs4 import BeautifulSoup
import json
import time
import re
import sys
import warnings
from bs4 import MarkupResemblesLocatorWarning

# ปิดการแจ้งเตือน BeautifulSoup สำหรับข้อความที่ไม่ใช่ HTML
warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

def scrape_thairath_banks(start_year, end_year, test_mode=False):
    """
    Scraper สำหรับ Thairath Money/Investment
    แก้ไข: เพิ่มระบบดึงวันที่แบบระบุวัน (YYYY-MM-DD) จาก URL หรือ Meta Tags
    """
    bank_map = [
        {"id": "KBANK", "keywords": ["KBANK", "ธนาคารกสิกรไทย", "กสิกรไทย"]},
        {"id": "SCB", "keywords": ["SCB", "ธนาคารไทยพาณิชย์", "ไทยพาณิชย์"]},
        {"id": "BBL", "keywords": ["BBL", "ธนาคารกรุงเทพ", "กรุงเทพ"]},
        {"id": "BAY", "keywords": ["BAY", "ธนาคารกรุงศรีอยุธยา", "กรุงศรี"]},
        {"id": "TTB", "keywords": ["TTB", "ธนาคารทหารไทยธนชาต", "ทหารไทยธนชาต"]},
        {"id": "KTB", "keywords": ["KTB", "ธนาคารกรุงไทย", "กรุงไทย"]}
    ]

    prefix = "test_" if test_mode else ""
    filename = f"{prefix}dataset_thairath_Banks_{start_year}_{end_year}.jsonl"

    print(f"\n{'='*80}")
    print(f"🚀 [START] Thairath Investment Scraper (Precise Date Mode) | Years: {start_year}-{end_year}")
    print(f"📂 Output File: {filename}")
    print(f"{'='*80}\n")

    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}
    total_saved = 0
    total_links_checked = 0

    with open(filename, "w", encoding="utf-8") as f_out:
        for year in range(start_year, end_year + 1):
            for month in range(1, 13):
                sitemap_url = f"https://www.thairath.co.th/sitemap-monthly-{year}-{month:02d}.xml"

                try:
                    print(f"🔍 [Sitemap] Checking: {year}-{month:02d}...")
                    resp = cureq.get(sitemap_url, headers=headers, impersonate="chrome", timeout=30)
                    if resp.status_code != 200: continue

                    soup = BeautifulSoup(resp.text, 'xml')
                    urls = [loc.text for loc in soup.find_all('loc') if '/money/investment/' in loc.text and 'gold' not in loc.text.lower()]

                    if not urls: continue
                    print(f"   🎯 Found {len(urls)} relevant links. Starting extraction...")

                    for idx, url in enumerate(urls, 1):
                        if test_mode and total_saved >= 10: break
                        total_links_checked += 1
                        sys.stdout.write(f"\r      [Processing] Link {idx}/{len(urls)} | Saved: {total_saved} | {url[-30:]}...")
                        sys.stdout.flush()

                        try:
                            # 1. พยายามดึงวันที่จาก URL ก่อน (รูปแบบ /2024/05/21/)
                            date_match = re.search(r'/(\d{4})/(\d{2})/(\d{2})/', url)
                            exact_date = f"{date_match.group(1)}-{date_match.group(2)}-{date_match.group(3)}" if date_match else f"{year}-{month:02d}-01"

                            art_resp = cureq.get(url, headers=headers, impersonate="chrome", timeout=15)
                            if art_resp.status_code != 200: continue
                            art_soup = BeautifulSoup(art_resp.text, 'html.parser')

                            # 2. ถ้ามี meta published_time ให้ใช้ตัวนั้นแทนเพื่อความแม่นยำ
                            meta_date = art_soup.find('meta', property='article:published_time')
                            if meta_date and meta_date.get('content'):
                                exact_date = meta_date['content'].split('T')[0]

                            title_tag = art_soup.find('h1')
                            title = title_tag.get_text(strip=True) if title_tag else ""
                            paragraphs = art_soup.find_all(['p', 'article'])
                            content_raw = ' '.join([p.get_text(strip=True) for p in paragraphs if len(p.get_text()) > 25])
                            content = re.sub(r'\s+', ' ', content_raw).strip()

                            if len(content) < 100: continue
                            full_text = f"{title} {content}".upper()

                            for bank_info in bank_map:
                                if any(kw.upper() in full_text for kw in bank_info["keywords"]):
                                    record = {
                                        "related_banks": bank_info["id"],
                                        "date": exact_date,
                                        "title": title,
                                        "content": content,
                                        "url": url
                                    }
                                    f_out.write(json.dumps(record, ensure_ascii=False) + "\n")
                                    total_saved += 1
                                    if test_mode and total_saved >= 10: break
                            time.sleep(0.05)
                        except: continue
                    print(f"\n   ✅ Month {year}-{month:02d} completed.")
                except Exception as e:
                    print(f"\n❌ Error in {year}-{month:02d}: {e}")
                    continue

    print(f"\n{'='*80}\n🏁 [FINISH] Total Saved: {total_saved} | File: {filename}\n{'='*80}")

if __name__ == '__main__':
    scrape_thairath_banks(2025, 2025, test_mode=False)
    scrape_thairath_banks(2023, 2024, test_mode=False)

In [ ]:
# [CELL-13]
# TNN Scraper with Bank Filtering Logic and Enhanced Logging
from curl_cffi import requests as cureq
from bs4 import BeautifulSoup
import json
import time
import re
import warnings
from tqdm.auto import tqdm
from bs4 import MarkupResemblesLocatorWarning

# ปิดการแจ้งเตือน BeautifulSoup สำหรับข้อความที่ไม่ใช่ HTML
warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

def ultra_refine_text(text):
    if not text: return ""
    ticker_patterns = [
        r'^.*?"ทรัมป์".*?"ซัมซุง".*?"Starbucks".*?สู้เงินเฟ้อ',
        r'^.*?Clean Energy:.*?YLG มองทองคำยังขาขึ้น',
        r'^.*?ราคาน้ำนมโลกลดลงต่อเนื่อง.*?จุดเสี่ยงเขย่าตลาดทองคำ'
    ]
    for pattern in ticker_patterns:
        text = re.sub(pattern, '', text).strip()

    footer_markers = [
        "TNN ทันโลก ทันเศรษฐกิจ", "อาคาร 101 ทรู ดิจิทัล",
        "Email :[email protected]", "WorldWealthHealthTechEarth"
    ]
    for marker in footer_markers:
        if marker in text: text = text.split(marker)[0].strip()

    text = text.replace('"', '')
    text = ' '.join(text.split())
    return text

def scrape_tnn_banks(start_year, end_year, test_mode=False):
    """
    Scraper สำหรับ TNN Wealth พร้อมระบบ Logging และ tqdm progress bar
    """
    bank_map = [
        {"id": "KBANK", "keywords": ["KBANK", "ธนาคารกสิกรไทย", "กสิกรไทย"]},
        {"id": "SCB", "keywords": ["SCB", "ธนาคารไทยพาณิชย์", "ไทยพาณิชย์"]},
        {"id": "BBL", "keywords": ["BBL", "ธนาคารกรุงเทพ", "กรุงเทพ"]},
        {"id": "BAY", "keywords": ["BAY", "ธนาคารกรุงศรีอยุธยา", "กรุงศรี"]},
        {"id": "TTB", "keywords": ["TTB", "ธนาคารทหารไทยธนชาต", "ทหารไทยธนชาต"]},
        {"id": "KTB", "keywords": ["KTB", "ธนาคารกรุงไทย", "กรุงไทย"]}
    ]

    prefix = "test_" if test_mode else ""
    filename = f"{prefix}dataset_tnn_Banks_{start_year}_{end_year}.jsonl"

    print(f"\n🚀 [Starting] TNN (Bank Specific Filtering) Year Range: {start_year}-{end_year}")
    session = cureq.Session(impersonate="chrome", timeout=30)

    target_months = []
    for y in range(end_year, start_year - 1, -1):
        for m in range(12, 0, -1):
            target_months.append(f"{y}{m:02d}")

    total_saved = 0
    with open(filename, "w", encoding="utf-8") as f:
        for yyyymm in target_months:
            sitemap_url = f"https://www.tnnthailand.com/sitemap_news_{yyyymm}.xml"
            try:
                resp = session.get(sitemap_url)
                if resp.status_code != 200: continue

                sm_soup = BeautifulSoup(resp.text, 'xml')
                links = [loc.text for loc in sm_soup.find_all('loc') if '/wealth/' in loc.text]

                if not links: continue

                print(f"\n📂 Processing {yyyymm} | Found {len(links)} candidate links")
                pbar = tqdm(links, desc=f"Month {yyyymm}", leave=False)

                for link in pbar:
                    if test_mode and total_saved >= 10: break
                    try:
                        art_resp = session.get(link)
                        if art_resp.status_code == 200:
                            soup = BeautifulSoup(art_resp.text, 'html.parser')

                            title_raw = soup.find('h1').get_text(strip=True) if soup.find('h1') else ""
                            content_div = soup.find('div', class_=re.compile(r'news-detail-content|content-detail|detail-box', re.I)) or soup.find('article')
                            paragraphs = [p.get_text(strip=True) for p in content_div.find_all('p')] if content_div else []
                            body_raw = " ".join(paragraphs)

                            title = ultra_refine_text(title_raw)
                            body = ultra_refine_text(body_raw)

                            if len(body) < 150: continue
                            full_text = f"{title} {body}".upper()

                            for bank_info in bank_map:
                                if any(kw.upper() in full_text for kw in bank_info["keywords"]):
                                    meta_date = soup.find('meta', property='article:published_time')
                                    record = {
                                        "related_banks": bank_info["id"],
                                        "date": meta_date['content'][:10] if meta_date else "",
                                        "title": title,
                                        "content": body,
                                        "url": link
                                    }
                                    f.write(json.dumps(record, ensure_ascii=False) + "\n")
                                    total_saved += 1
                                    pbar.set_postfix({"saved": total_saved})
                                    if test_mode and total_saved >= 10: break

                        time.sleep(0.1)
                    except: continue
            except Exception as e:
                print(f"\n❌ Error processing month {yyyymm}: {e}")

            if test_mode and total_saved >= 10: break

    print(f"\n✅ Scrape Finished! Total saved: {total_saved} records. File: {filename}")

if __name__ == '__main__':
    TEST_MODE = False
    # แยกปีเพื่อป้องกัน Memory/Connection issue
    scrape_tnn_banks(2025, 2025, test_mode=TEST_MODE)
    scrape_tnn_banks(2023, 2024, test_mode=TEST_MODE)

In [ ]:
# [CELL-14] TNN cleaner
import json
import re
import os
from tqdm.auto import tqdm

def final_refine_content(text):
    if not text: return ""

    # 1. กำจัด Dynamic Gold & Market Junk (Safe patterns)
    gold_junk_patterns = [
        r'^(ข่าวแนะนำ|อ่านข่าว|ทองขึ้น|ทองลง).*?Sideways',
        r'^ทอง(ขึ้น|ลง)\s*[+-]\$\d+\.\d+.*?Sideways'
    ]
    for p in gold_junk_patterns:
        text = re.sub(p, '', text, flags=re.IGNORECASE | re.DOTALL)

    # 2. DISABLED: Quote-based removal (ปิดถาวรเพื่อความปลอดภัยสูงสุด)
    # text = re.sub(r'^(["\x27\u201c\u2018][^"\x27\u201d\u2019]{1,50}["\x27\u201d\u2019]\s*){2,3}', '', text)

    # 3. FIXED: ปรับปรุงการลบ 'สรุปข่าว' ให้ลบเฉพาะคำนำหน้าสั้นๆ (ไม่เกิน 20 ตัวอักษร)
    # เพื่อไม่ให้ไปกินเนื้อหาหลักที่ตามมา
    text = re.sub(r'^สรุปข่าว[:\s]{1,10}', '', text)

    # 4. FIXED: Footer Removal (ลบเฉพาะบรรทัดเครดิต)
    credit_patterns = [
        r'(ขอบคุณข้อมูลจาก|ที่มา|เครดิตภาพ|ภาพจาก|เรียบเรียงโดย|Source|Credit|ที่มาข่าว|สำนักข่าว|รายงานโดย).*',
        r'ห้ามคัดลอก.*', r'ติดตามข้อมูลเพิ่มเติม.*', r'ช่องทางติดตาม.*',
        r'Add Line.*', r'กดติดตาม.*', r'Youtube.*TikTok.*Facebook.*',
        r'สนใจสมัคร.*', r'อ่านข่าวต่อได้ที่.*'
    ]
    for pattern in credit_patterns:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)

    # 5. จัดการช่องว่าง
    text = text.replace('\xa0', ' ')
    text = re.sub(r'\s+', ' ', text).strip()

    return text

def process_tnn_files_only():
    tnn_files = ["dataset_tnn_Banks_2023_2024.jsonl", "dataset_tnn_Banks_2025_2025.jsonl"]

    for input_file in tnn_files:
        if not os.path.exists(input_file): continue
        output_file = input_file.replace('.jsonl', '_cleaned.jsonl')
        print(f"\n🧹 Re-cleaning TNN (Recovery Mode): {input_file}")
        count = 0
        with open(input_file, 'r', encoding='utf-8') as f:
            lines = f.readlines()
        with open(output_file, 'w', encoding='utf-8') as f_out:
            for line in tqdm(lines, desc=f"Processing {input_file}", leave=False):
                try:
                    data = json.loads(line)
                    cleaned = final_refine_content(data.get('content', ''))
                    data['content'] = cleaned
                    if len(cleaned) > 100:
                        f_out.write(json.dumps(data, ensure_ascii=False) + '\n')
                        count += 1
                except: continue
        print(f"✅ Finished: {output_file} ({count} records)")

if __name__ == "__main__":
    process_tnn_files_only()
    print("\n✨ [Update] แก้ไข Regex 'สรุปข่าว' และกู้คืนข้อมูลปี 2023-2024 เรียบร้อยแล้วครับ!")

In [ ]:
# [CELL-15]
# Share2trade Scraper (v1.6 Manual Page Control & High-Speed Edition)
from curl_cffi import requests as cureq
from bs4 import BeautifulSoup
import json
import time
import re
import sys
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed

warnings.filterwarnings("ignore")

def scrape_share2trade_banks(start_year=2023, end_year=2024, test_mode=False):
    """
    Scraper สำหรับ Share2trade ที่ผู้ใช้กำหนดจุดหยุด (Stop Page) ได้เอง
    """
    bank_map = [
        {"id": "KBANK", "keywords": ["KBANK", "ธนาคารกสิกรไทย", "กสิกรไทย"]},
        {"id": "SCB", "keywords": ["SCB", "ธนาคารไทยพาณิชย์", "ไทยพาณิชย์"]},
        {"id": "BBL", "keywords": ["BBL", "ธนาคารกรุงเทพ", "กรุงเทพ"]},
        {"id": "BAY", "keywords": ["BAY", "ธนาคารกรุงศรีอยุธยา", "กรุงศรี"]},
        {"id": "TTB", "keywords": ["TTB", "ธนาคารทหารไทยธนชาต", "ทหารไทยธนชาต"]},
        {"id": "KTB", "keywords": ["KTB", "ธนาคารกรุงไทย", "กรุงไทย"]}
    ]

    prefix = "test_" if test_mode else ""
    filename = f"{prefix}dataset_share2trade_Banks_{start_year}_{end_year}.jsonl"

    # ⚙️ CONFIGURATION: ตั้งค่าหน้าที่จะให้เริ่มและหยุดของแต่ละหมวดหมู่ที่นี่
    categories_config = [
        {"path": "/news/talk-of-the-town", "name": "Talk of the Town", "start_page": 1, "stop_page": 214},
        {"path": "/news/bulletin-board", "name": "Bulletin Board", "start_page": 1, "stop_page": 438},
        {"path": "/news/stock-daily-journal", "name": "Stock Daily Journal", "start_page": 1, "stop_page": 39},
        {"path": "/news/premium-stock", "name": "Premium Stock", "start_page": 1, "stop_page": 39}
    ]

    print(f"\n{'='*80}")
    print(f"🚀 [START] Share2trade Bank Scraper v1.6 (Manual Page Control)")
    print(f"📂 Output: {filename} | Targets: {start_year}-{end_year}")
    print(f"{'='*80}\n")

    base_url = "https://www.share2trade.com"
    session = cureq.Session(impersonate="chrome", timeout=30)
    total_saved = 0

    def fetch_article_content(link):
        try:
            full_link = f"{base_url}{link}" if link.startswith('/') else link
            r = session.get(full_link, timeout=20)
            if r.status_code != 200: return None
            soup = BeautifulSoup(r.text, 'html.parser')
            detail_zone = soup.find(class_=re.compile(r'news-detail|article|content', re.I)) or soup
            date_el = detail_zone.find(class_=re.compile(r'date|time', re.I))
            date_text = date_el.get_text(strip=True) if date_el else ""
            year_match = re.search(r'(20\d{2}|25\d{2})', date_text)
            if not year_match: return None
            yr = int(year_match.group())
            disp_yr = yr - 543 if yr > 2500 else yr
            body_tag = soup.find('div', class_=re.compile(r'news-detail__content|content-body|entry-content', re.I)) or detail_zone
            valid_texts = [p.get_text(strip=True) for p in body_tag.find_all(['p', 'div'], recursive=False) if len(p.get_text(strip=True)) > 15]
            text = re.sub(r'\s+', ' ', " ".join(valid_texts)).strip()
            if len(text) < 80: return None
            title = soup.find('h1').get_text(strip=True) if soup.find('h1') else ""
            return {"title": title, "content": text, "date": date_text, "url": full_link, "year": disp_yr}
        except: return None

    try:
        with open(filename, "w", encoding="utf-8") as f_out:
            for cat in categories_config:
                print(f"\n📂 Category: {cat['name']} | Limit: Page {cat['start_page']} to {cat['stop_page']}")
                page = cat['start_page']
                prev_links_hash = ""

                while True:
                    if cat['stop_page'] and page > cat['stop_page']:
                        print(f"\n   🛑 Manual stop reached at page {page-1}.")
                        break

                    target = f"{base_url}{cat['path']}?page={page}"
                    try:
                        resp = session.get(target, timeout=20)
                        if resp.status_code != 200: break
                    except: break

                    soup = BeautifulSoup(resp.text, 'html.parser')
                    links = [a['href'] for a in soup.select('div.card__body a[href]')]
                    current_hash = "".join(sorted(links))
                    if not links or current_hash == prev_links_hash:
                        print(f"\n   🏁 Content ended or duplicated at page {page}.")
                        break
                    prev_links_hash = current_hash

                    page_years = []
                    with ThreadPoolExecutor(max_workers=10) as executor:
                        futures = {executor.submit(fetch_article_content, l): l for l in links}
                        for future in as_completed(futures):
                            res = future.result()
                            if not res: continue
                            page_years.append(res['year'])
                            if start_year <= res['year'] <= end_year:
                                full_text = f"{res['title']} {res['content']}".upper()
                                for bank in bank_map:
                                    if any(kw.upper() in full_text for kw in bank["keywords"]):
                                        save_data = {"related_banks": bank["id"], "date": res['date'], "title": res['title'], "content": res['content'], "url": res['url']}
                                        f_out.write(json.dumps(save_data, ensure_ascii=False) + "\n")
                                        total_saved += 1
                                        break

                    range_text = f"{min(page_years)}-{max(page_years)}" if page_years else "N/A"
                    sys.stdout.write(f"\r   📖 Page {page}/{cat['stop_page']} | Year Range: {range_text} | Total Saved: {total_saved}")
                    sys.stdout.flush()
                    f_out.flush()
                    page += 1
                    time.sleep(0.05)

    except KeyboardInterrupt:
        print("\n\n⚠️ Stopped by user. Progress saved.")
    finally:
        session.close()
        print(f"\n{'='*80}\n🏁 [FINISH] Total Bank Records: {total_saved} | File: {filename}\n{'='*80}")

if __name__ == "__main__":
    # แยกไฟล์ปีปัจจุบัน 2025
    scrape_share2trade_banks(2025, 2025, test_mode=False)
    # แยกไฟล์ข่าวย้อนหลัง 2023-2024
    scrape_share2trade_banks(2023, 2024, test_mode=False)

In [ ]:
# [CELL-16]
# Prachachat Scraper with Hybrid Bank Filtering & Advanced Logging
import requests
import json
import time
import re
import random
import sys
from bs4 import BeautifulSoup

def fetch_and_parse(url, timeout=20):
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
    try:
        resp = requests.get(url, headers=headers, timeout=timeout)
        if resp.status_code == 200:
            return BeautifulSoup(resp.text, 'html.parser')
    except Exception:
        pass
    return None

def scrape_prachachat_banks(target_start_year=2023, target_end_year=2024, start_page=1, test_mode=False):
    """
    Scraper สำหรับ Prachachat Finance พร้อมระบบคัดกรองรายธนาคาร
    """
    bank_map = [
        {"id": "KBANK", "keywords": ["KBANK", "ธนาคารกสิกรไทย", "กสิกรไทย"]},
        {"id": "SCB", "keywords": ["SCB", "ธนาคารไทยพาณิชย์", "ไทยพาณิชย์"]},
        {"id": "BBL", "keywords": ["BBL", "ธนาคารกรุงเทพ", "กรุงเทพ"]},
        {"id": "BAY", "keywords": ["BAY", "ธนาคารกรุงศรีอยุธยา", "กรุงศรี"]},
        {"id": "TTB", "keywords": ["TTB", "ธนาคารทหารไทยธนชาต", "ทหารไทยธนชาต"]},
        {"id": "KTB", "keywords": ["KTB", "ธนาคารกรุงไทย", "กรุงไทย"]}
    ]

    prefix = "test_" if test_mode else ""
    output_filename = f"{prefix}dataset_prachachat_Banks_{target_start_year}_{target_end_year}.jsonl"

    print(f"\n{'='*80}")
    print(f"🚀 [START] Prachachat Finance Scraper | Range: {target_start_year}-{target_end_year}")
    print(f"📂 Output File: {output_filename}")
    print(f"{'='*80}\n")

    page = start_page
    total_saved = 0
    consecutive_empty_pages = 0

    with open(output_filename, 'w', encoding='utf-8') as f_out:
        while True:
            current_page_url = f"https://www.prachachat.net/finance/page/{page}"
            print(f"🔍 [Page {page}] Scanning: {current_page_url}")

            soup = fetch_and_parse(current_page_url)
            if not soup:
                print(f"   ⚠️ Load failed. Skipping page {page}")
                page += 1
                continue

            # Extract Article Links
            article_links = []
            for a in soup.find_all('a', href=True):
                href = a['href']
                if '/news-' in href and ('/finance/' in href or 'prachachat.net' in href):
                    if href.startswith('/'): href = 'https://www.prachachat.net' + href
                    article_links.append(href)

            article_links = list(set(article_links))

            if not article_links:
                consecutive_empty_pages += 1
                if consecutive_empty_pages > 20:
                    print("   🛑 No more news found. Ending process.")
                    break
                page += 1
                continue

            consecutive_empty_pages = 0
            print(f"   🎯 Found {len(article_links)} links. Extracting content...")

            for idx, article_url in enumerate(article_links, 1):
                if test_mode and total_saved >= 10: break

                sys.stdout.write(f"\r      [Processing] {idx}/{len(article_links)} | Total Saved: {total_saved} | Link: {article_url[:40]}...")
                sys.stdout.flush()

                art_soup = fetch_and_parse(article_url)
                if not art_soup: continue

                # Date Parsing & Year Conversion
                date_tag = art_soup.find('time', class_='entry-date')
                if not date_tag or not date_tag.get('datetime'): continue

                raw_date = date_tag.get('datetime').split('T')[0]
                year = int(raw_date.split('-')[0])
                if year > 2500: year -= 543

                # Stop Condition: ถ้าเจอปีเก่ากว่าที่กำหนด
                if year < target_start_year:
                    print(f"\n   🛑 Reached year {year}. Stopping.")
                    return

                if target_start_year <= year <= target_end_year:
                    title_tag = art_soup.find('h1')
                    title = title_tag.get_text(strip=True) if title_tag else ""

                    content_div = art_soup.find('div', class_='td-post-content') or art_soup.find('div', class_='entry-content')
                    if not content_div: continue

                    paragraphs = [p.get_text(strip=True) for p in content_div.find_all('p') if len(p.get_text(strip=True)) > 20]
                    body = ' '.join(paragraphs)
                    clean_body = re.sub(r'\s+', ' ', body).strip()

                    full_text = f"{title} {clean_body}".upper()

                    # Hybrid Filtering
                    for bank_info in bank_map:
                        if any(kw.upper() in full_text for kw in bank_info["keywords"]):
                            record = {
                                "related_banks": bank_info["id"],
                                "date": f"{year}-{raw_date.split('-', 1)[1]}",
                                "title": title,
                                "content": clean_body,
                                "url": article_url
                            }
                            f_out.write(json.dumps(record, ensure_ascii=False) + '\n')
                            total_saved += 1
                            if test_mode and total_saved >= 10: break

                time.sleep(random.uniform(0.1, 0.2))

            print(f"\n   ✅ Page {page} processed.")
            page += 1

    print(f"\n{'='*80}")
    print(f"🏁 [FINISH] Total Banks Records Saved: {total_saved}")
    print(f"📂 File: {output_filename}")
    print(f"{'='*80}\n")

if __name__ == "__main__":
    TEST_MODE = False
    # ดึงข้อมูลปี 2025 (เริ่มจากหน้า 1)
    scrape_prachachat_banks(2025, 2025, start_page=190, test_mode=TEST_MODE)
    # ดึงข้อมูลปี 2023-2024 (เริ่มจากหน้า 835 ตามตำแหน่ง Archive)
    scrape_prachachat_banks(2023, 2024, start_page=835, test_mode=TEST_MODE)

In [ ]:
# [CELL-17]
from curl_cffi import requests as cureq
from bs4 import BeautifulSoup
import json
import re
import time
import random
import threading
import warnings
from bs4 import MarkupResemblesLocatorWarning
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

IMPERSONATE_PROFILES = ["chrome100", "chrome110", "chrome120", "edge99", "edge101", "safari15_3", "safari15_5"]

# --- 🛡️ ระบบจัดการ Session ประจำ Thread (แก้ปัญหาท่อเต็ม) ---
thread_local = threading.local()

def get_session():
    if not hasattr(thread_local, "session"):
        # สร้าง Session และจำลองตัวตนเบราว์เซอร์ 1 แบบ ประจำ Worker นี้เลย
        thread_local.session = cureq.Session(impersonate=random.choice(IMPERSONATE_PROFILES))
    return thread_local.session

def extract_exact_date(text, fallback_date):
    thai_months = {
        "ม.ค.": "01", "ก.พ.": "02", "มี.ค.": "03", "เม.ย.": "04",
        "พ.ค.": "05", "มิ.ย.": "06", "ก.ค.": "07", "ส.ค.": "08",
        "ก.ย.": "09", "ต.ค.": "10", "พ.ย.": "11", "ธ.ค.": "12"
    }
    match = re.search(r'(\d{1,2})\s*(ม\.ค\.|ก\.พ\.|มี\.ค\.|เม\.ย\.|พ\.ค\.|มิ\.ย\.|ก\.ค\.|ส\.ค\.|ก\.ย\.|ต\.ค\.|พ\.ย\.|ธ\.ค\.)\s*(\d{2,4})', text)

    if match:
        d = int(match.group(1))
        m_str = match.group(2)
        y = int(match.group(3))
        m = thai_months.get(m_str, "01")
        if y < 100: y += 2500
        if y > 2500: y -= 543
        return f"{y}-{m}-{d:02d}"
    return fallback_date + "-01" if len(fallback_date) == 7 else fallback_date


# --- 1. Worker Function (อัปเกรดระบบ Auto-Retry 3 ครั้ง) ---
def process_single_article(args):
    url, sitemap_date, bank_map = args
    session = get_session() # ดึง Session ประจำตัวมาใช้

    # 🔄 ระบบตื้อ (Retry Loop)
    for attempt in range(3):
        try:
            # พักหายใจก่อนยิงเสมอ
            time.sleep(random.uniform(0.5, 1.5))
            art_resp = session.get(url, timeout=20)

            # ถ้าเว็บตอบกลับปกติ ให้ออกจาก Loop ตื้อทันที
            if art_resp.status_code == 200:
                break
            # ถ้าโดนบล็อก ให้พักเบรกยาวหน่อย แล้วลองใหม่
            elif art_resp.status_code in [403, 429, 502, 503]:
                time.sleep(3)
                continue

        except Exception:
            time.sleep(2)
            continue
    else:
        # ถ้าพยายามครบ 3 รอบแล้วยังพังอยู่ ให้ยอมแพ้
        return {"status": "error", "error_code": "Max_Retries_Exceeded"}

    # --- ส่วนดึงเนื้อหาตามปกติ ---
    art_resp.encoding = 'utf-8'
    soup = BeautifulSoup(art_resp.content, 'html.parser')

    title_tag = soup.find('h1') or soup.find(id='hlTitle') or soup.find('title')
    title = title_tag.get_text(strip=True).replace('"', '') if title_tag else ""

    if any(blacklist in title.lower() for blacklist in ["login", "เข้าสู่ระบบ", "efin stockpickup", "member", "just a moment"]):
        return {"status": "blocked"}

    content_box = soup.find(id=re.compile(r'^(spnDetail|divDetail|divNewsDetail)$', re.IGNORECASE))
    text = ""

    if content_box:
        for junk in content_box.find_all(['script', 'style', 'iframe']):
            junk.decompose()
        text = content_box.get_text(separator=' ', strip=True)

    if not text or len(text) < 150:
        paragraphs = soup.find_all(['div', 'p'])
        all_texts = [p.get_text(separator=' ', strip=True) for p in paragraphs]
        if all_texts:
            text = max(all_texts, key=len)

    exact_date = sitemap_date
    if len(sitemap_date) < 10:
        raw_full_text = soup.get_text(separator=' ')
        exact_date = extract_exact_date(raw_full_text, sitemap_date)

    text = re.sub(r'หน้าหลัก\s*ข่าวหุ้นล่าสุด\s*รายละเอียด\s*ข่าวหุ้นล่าสุด', '', text)
    text = re.sub(r'Share\s*สำนักข่าวอีไฟแนนซ์ไทย[^\n]{0,50}น\.', '', text)
    text = re.sub(r'สำนักข่าวอีไฟแนนซ์ไทย\s*-\s*-', '', text)
    text = re.sub(r'Light Mode.*?Swift Mode', '', text, flags=re.S)
    text = re.sub(r'\s+', ' ', text).strip().replace('"', '')

    if len(text) < 150 or "ลืมรหัสผ่าน" in text or "Username" in text:
        return {"status": "too_short"}

    full_text = f"{title} {text}".upper()
    results = []
    found_bank = False

    for bank_info in bank_map:
        if any(kw.upper() in full_text for kw in bank_info["keywords"]):
            results.append({
                "related_banks": bank_info["id"],
                "date": exact_date,
                "title": title,
                "content": text,
                "url": url
            })
            found_bank = True

    if found_bank:
        return {"status": "saved", "records": results}
    else:
        return {"status": "no_keyword"}


# --- 2. Main Orchestrator ---
def scrape_efin_banks_fast(start_year, end_year, max_workers=5, test_mode=False):
    bank_map = [
        {"id": "KBANK", "keywords": ["KBANK", "ธนาคารกสิกรไทย", "กสิกรไทย"]},
        {"id": "SCB", "keywords": ["SCB", "ธนาคารไทยพาณิชย์", "ไทยพาณิชย์"]},
        {"id": "BBL", "keywords": ["BBL", "ธนาคารกรุงเทพ", "กรุงเทพ"]},
        {"id": "BAY", "keywords": ["BAY", "ธนาคารกรุงศรีอยุธยา", "กรุงศรี"]},
        {"id": "TTB", "keywords": ["TTB", "ธนาคารทหารไทยธนชาต", "ทหารไทยธนชาต"]},
        {"id": "KTB", "keywords": ["KTB", "ธนาคารกรุงไทย", "กรุงไทย"]}
    ]

    filename = f"dataset_efinancethai_Banks_{start_year}_{end_year}.jsonl"

    print(f"\n{'='*80}")
    print(f"🚀 [START] Armored Scraper | {start_year}-{end_year}")
    print(f"{'='*80}\n")

    all_tasks = []
    print("🔍 Step 1: Scanning Sitemaps for targets...")
    session = cureq.Session(impersonate="chrome110")
    for year in range(start_year, end_year + 1):
        for month in range(1, 13):
            if test_mode and month > 1: break

            sitemap_url = f"https://www.efinancethai.com/sitemap/sitemaps-news/sitemap-news-{year}-{month:02d}.xml"
            try:
                resp = session.get(sitemap_url, timeout=15)
                if resp.status_code == 200:
                    soup_sm = BeautifulSoup(resp.text, 'xml')
                    for url_node in soup_sm.find_all('url'):
                        loc_node = url_node.find('loc')
                        if loc_node and "LatestNewsMain" in loc_node.text:
                            lastmod_node = url_node.find('lastmod')
                            sdate = lastmod_node.text[:10] if lastmod_node and len(lastmod_node.text)>=10 else f"{year}-{month:02d}"
                            all_tasks.append((loc_node.text, sdate, bank_map))
            except Exception:
                continue

    print(f"🎯 Found {len(all_tasks):,} links. Starting Extraction (Threads: {max_workers})...\n")

    stats = {"scanned": 0, "saved": 0, "no_keyword": 0, "too_short": 0, "blocked": 0, "errors": 0}

    with open(filename, "w", encoding="utf-8") as f_out:
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = [executor.submit(process_single_article, task) for task in all_tasks]

            for future in tqdm(as_completed(futures), total=len(futures), desc="Extracting News"):
                res = future.result()
                stats["scanned"] += 1

                if res["status"] == "saved":
                    for rec in res["records"]:
                        tqdm.write(f"   🟢 [SAVE] {rec['related_banks']} | {rec['date']} | {rec['title'][:40]}...")
                        f_out.write(json.dumps(rec, ensure_ascii=False) + "\n")
                        stats["saved"] += 1
                elif res["status"] == "no_keyword":
                    stats["no_keyword"] += 1
                elif res["status"] == "too_short":
                    stats["too_short"] += 1
                elif res["status"] == "blocked":
                    stats["blocked"] += 1
                elif res["status"] == "error":
                    stats["errors"] += 1

                if stats["scanned"] % 500 == 0:
                    tqdm.write(f"\n📊 [LIVE STATS] ผ่านไป {stats['scanned']} ลิงก์ 👉 เซฟแบงก์: {stats['saved']} | ข่าวทั่วไป: {stats['no_keyword']} | หน้าสั้น: {stats['too_short']} | โดนบล็อก: {stats['blocked']} | Errors: {stats['errors']}\n")

    print(f"\n{'='*80}")
    print(f"🏁 [FINISH SUMMARY]")
    print(f"  ✅ ได้ข่าวธนาคาร: {stats['saved']:,} บทความ")
    print(f"  🗑️ ข่าวทั่วไป: {stats['no_keyword']:,} ลิงก์")
    print(f"  ❌ Errors (ทิ้งจริงๆ): {stats['errors']:,} ลิงก์")
    print(f"{'='*80}\n")

if __name__ == "__main__":
    TEST_MODE = False
    # ลด Workers เหลือ 5 ตัวเพื่อไม่ให้เซิร์ฟเวอร์ตกใจ
    scrape_efin_banks_fast(2023, 2024, max_workers=5, test_mode=TEST_MODE)
    scrape_efin_banks_fast(2025, 2025, max_workers=5, test_mode=TEST_MODE)

In [ ]:
# [CELL-19]
import pandas as pd
import glob
import json
import os
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

def clean_and_parse_dates(date_series):
    s = date_series.astype(str)
    clean_date_str = s.str.extract(r'(\d{4}-\d{2}-\d{2})')[0]
    return pd.to_datetime(clean_date_str, format='%Y-%m-%d', errors='coerce')

def run_comprehensive_statistical_summary(target_folders):
    all_data = []
    core_sources = {
        'Thunhoon': ['thunhoon', 'thun'],
        'Kaohoon': ['kaohoon', 'kao'],
        'Hoonsmart': ['hoonsmart'],
        'Thaipr': ['thaipr'],
        'Money & Banking': ['money_banking', 'moneyandbanking', 'money'],
        'PPTV': ['pptv'],
        'Infoquest': ['infoquest', 'iq'],
        'Amarin': ['amarin'],
        'Thairath': ['thairath'],
        'TNN': ['tnn'],
        'BusinessToday': ['businesstoday', 'bt'],
        'HoonVision': ['hoonvision', 'vision'],
        'eFinanceThai': ['efinancethai', 'efin'],
        'Prachachat': ['Prachachat'],
        'Share2Trade': ['Share2Trade']
    }

    print(f"Loading data from {len(target_folders)} folders...")

    for period, folder_path in target_folders.items():
        try:
            if not os.path.exists(folder_path): continue
            files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.startswith("dataset")]
        except Exception as e:
            print(f"Error accessing folder: {folder_path} | {e}")
            continue

        for f in tqdm(files, desc=f"Processing {period}"):
            source_tag = "Other"
            fname_lower = os.path.basename(f).lower()
            for display_name, keywords in core_sources.items():
                if any(kw in fname_lower for kw in keywords):
                    source_tag = display_name
                    break
            try:
                with open(f, 'r', encoding='utf-8-sig', errors='replace') as file:
                    for line in file:
                        line = line.strip()
                        if not line: continue
                        try:
                            item = json.loads(line)
                            item['source_category'] = source_tag
                            item['period_group'] = period
                            all_data.append(item)
                        except json.JSONDecodeError: continue
            except Exception: pass

    if not all_data:
        print("No data loaded. Please check folder paths.")
        return None

    df = pd.DataFrame(all_data)
    df['parsed_date'] = clean_and_parse_dates(df['date'])
    df = df.dropna(subset=['parsed_date'])
    df['year'] = df['parsed_date'].dt.year
    df['month'] = df['parsed_date'].dt.month

    # Filter strictly for 2023-2025 scope
    df = df[df['year'].isin([2023, 2024, 2025])].copy()

    print("\n" + "="*80)
    print("PART 1: GLOBAL OVERVIEW (2023-2025 Scope)")
    print("="*80)
    print(f"Total valid records: {len(df):,} records\n")

    global_stats = pd.crosstab(df['related_banks'], df['year'], margins=True)
    display(global_stats)

    plt.figure(figsize=(10, 5))
    sns.countplot(data=df, x='related_banks', hue='year', palette='viridis')
    plt.title('News Distribution per Bank (Thesis Scope: 2023-2025)')
    plt.show()

    print("\n" + "="*80)
    print("PART 2: INDIVIDUAL BANK ANALYSIS (Monthly Flow)")
    print("="*80)
    for bank in sorted(df['related_banks'].unique()):
        print(f"\nBank: {bank}")
        bank_df = df[df['related_banks'] == bank]
        pivot = pd.pivot_table(bank_df, values='title', index='year', columns='month', aggfunc='count', fill_value=0)
        fig, ax = plt.subplots(figsize=(12, 3))
        sns.heatmap(pivot, annot=True, fmt="d", cmap="YlGnBu", ax=ax)
        ax.set_title(f'News Density Heatmap: {bank} (2023-2025)')
        plt.show()

    print("\n" + "="*80)
    print("PART 3: SOURCE ANALYSIS")
    print("="*80)
    source_bank_stats = pd.crosstab(df['related_banks'], df['source_category'])
    source_bank_stats.plot(kind='bar', stacked=True, figsize=(12, 6), colormap='Set3')
    plt.title('News Source Composition (2023-2025)')
    plt.legend(title='Source', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()
    return df

target_news_folders = {
    "2023-2024": r"C:\Users\USER\Desktop\BANKS_News\Final\ForFinetuning",
    "2025": r"C:\Users\USER\Desktop\BANKS_News\Final\ForAnalyze"
}

final_analysis_df = run_comprehensive_statistical_summary(target_news_folders)